# Reproduce `discovered_cts_boxplot_0.1.png`

This notebook contains only the inputs, analysis, and plotting code needed for the **later** figure in the source notebook that writes `discovered_cts_boxplot_0.1.png` (the figure titled *Marginal cell type-disease associations*).

**Input**

- One scDRS cell-type result file per trait in `tms_data/scdrs_results/`
- Each file must be named `<trait>.scdrs_ct.cell_ontology_class`
- Each file must contain an `assoc_mcp` column

**Output**

- `discovered_cts_boxplot_0.1.png`

No AnnData object, expression matrix, Scanpy processing, or unrelated figures are required.

## 1. Imports and analysis settings

The original figure applies Benjamini-Hochberg correction separately within each trait and calls a cell type associated when its FDR is below 0.05.

In [1]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from statsmodels.stats.multitest import multipletests

RESULTS_DIR = RESULTS / "ct" / "tms_facs_none_ctrl"
RESULT_SUFFIX = ".scdrs_ct.cell_ontology_class"
OUTPUT_FILE = Path("discovered_cts_boxplot_0.1.png")

FDR_THRESHOLD = 0.05
JITTER_SEED = 42

## 2. Trait groups and eligible cell types

For each trait, the plot counts only associated cell types that belong to the corresponding curated brain or immune list.

In [3]:
BRAIN_TRAITS = [
    "UKB_460K.cov_EDU_YEARS",
    "PASS_SWB",
    "PASS_ReactionTime_Davies2018",
    "UKB_460K.cov_EDU_COLLEGE",
    "PASS_Worry_Nagel2018",
    "PASS_Insomnia_Jansen2019",
    "PASS_BIP_Mullins2021",
    "PASS_MDD_Howard2019",
    "UKB_460K.mental_NEUROTICISM",
    "PASS_ADHD_Demontis2018",
    "PASS_Schizophrenia_Pardinas2018",
    "UKB_460K.other_MORNINGPERSON",
    "PASS_Intelligence_SavageJansen2018",
    "PASS_VerbalNumericReasoning_Davies2018",
]

IMMUNE_TRAITS = [
    "UKB_460K.disease_AID_ALL",
    "PASS_Rheumatoid_Arthritis",
    "UKB_460K.blood_LYMPHOCYTE_COUNT",
    "UKB_460K.blood_EOSINOPHIL_COUNT",
    "PASS_Primary_biliary_cirrhosis",
    "PASS_Celiac",
    "PASS_Lupus",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "PASS_Multiple_sclerosis",
    "PASS_CD_deLange2017",
    "UKB_460K.blood_WHITE_COUNT",
    "PASS_IBD_deLange2017",
    "UKB_460K.blood_MONOCYTE_COUNT",
    "PASS_UC_deLange2017",
]

BRAIN_CELL_TYPES = [
    "oligodendrocyte",
    "microglial cell",
    "astrocyte",
    "ependymal cell",
    "brain pericyte",
    "interneuron",
    "neuron",
    "oligodendrocyte precursor cell",
    "Bergmann glial cell",
    "neuroepithelial cell",
    "neuronal stem cell",
    "medium spiny neuron",
]

IMMUNE_CELL_TYPES = [
    "B cell",
    "CD4-positive, alpha-beta T cell",
    "T cell",
    "CD8-positive, alpha-beta T cell",
    "leukocyte",
    "epithelial cell of thymus",
    "macrophage",
    "monocyte",
    "thymocyte",
    "NK cell",
    "mature NK T cell",
    "myeloid cell",
    "naive B cell",
    "late pro-B cell",
    "immature B cell",
    "myeloid leukocyte",
    "myeloid dendritic cell",
    "Kupffer cell",
    "precursor B cell",
    "neutrophil",
    "hematopoietic stem cell",
    "classical monocyte",
    "plasma cell",
    "plasmacytoid dendritic cell",
    "regulatory T cell",
    "dendritic cell",
    "non-classical monocyte",
    "promonocyte",
    "early pro-B cell",
    "intermediate monocyte",
    "lymphocyte",
    "mature alpha-beta T cell",
    "basophil",
    "granulocyte monocyte progenitor cell",
    "granulocytopoietic cell",
    "professional antigen presenting cell",
    "lymphoid progenitor cell",
    "DN4 thymocyte",
    "lung macrophage",
]

CATEGORIES = {
    "Brain": {
        "traits": BRAIN_TRAITS,
        "cell_types": BRAIN_CELL_TYPES,
    },
    "Immune": {
        "traits": IMMUNE_TRAITS,
        "cell_types": IMMUNE_CELL_TYPES,
    },
}

## 3. Count associated cell types for each trait

Zero-discovery traits are retained in the complete count table for transparency, then excluded from the plotted distributions to match the source figure.

In [4]:
def count_associated_cell_types(
    trait: str,
    eligible_cell_types: list[str],
) -> int:
    """Return the number of eligible cell types associated with one trait."""
    result_file = RESULTS_DIR / f"{trait}{RESULT_SUFFIX}"
    if not result_file.is_file():
        raise FileNotFoundError(
            f"Missing scDRS result for {trait!r}: {result_file}"
        )

    score_df = pd.read_csv(result_file, sep="\t", index_col=0)
    if "assoc_mcp" not in score_df.columns:
        raise KeyError(f"{result_file} does not contain an 'assoc_mcp' column")

    p_values = pd.to_numeric(score_df["assoc_mcp"], errors="raise").to_numpy()
    if not np.isfinite(p_values).all():
        raise ValueError(f"{result_file} contains non-finite assoc_mcp values")

    fdr = multipletests(p_values, method="fdr_bh")[1]
    associated_cell_types = score_df.index[fdr < FDR_THRESHOLD]

    # Match the source notebook: count associated names found in the curated list.
    return int(associated_cell_types.isin(eligible_cell_types).sum())


count_records = []
for category, specification in CATEGORIES.items():
    for trait in specification["traits"]:
        count_records.append(
            {
                "category": category,
                "trait": trait,
                "discovered_cell_types": count_associated_cell_types(
                    trait,
                    specification["cell_types"],
                ),
            }
        )

counts_df = pd.DataFrame(count_records)
plot_counts_df = counts_df.loc[
    counts_df["discovered_cell_types"] > 0
].copy()

counts_df

,category,trait,discovered_cell_types
0,Brain,UKB_460K.cov_EDU_YEARS,6
1,Brain,PASS_SWB,7
2,Brain,PASS_ReactionTime_Davies2018,7
3,Brain,UKB_460K.cov_EDU_COLLEGE,6
4,Brain,PASS_Worry_Nagel2018,5
5,Brain,PASS_Insomnia_Jansen2019,0
6,Brain,PASS_BIP_Mullins2021,5
7,Brain,PASS_MDD_Howard2019,8
8,Brain,UKB_460K.mental_NEUROTICISM,5
9,Brain,PASS_ADHD_Demontis2018,4


## 4. Compute the plotted means and 95% confidence intervals

The confidence interval matches the source notebook: `mean ± 1.96 × sample standard error`.

In [5]:
def mean_and_ci95(values: pd.Series) -> tuple[float, float]:
    """Return the mean and normal-approximation 95% CI half-width."""
    x = values.to_numpy(dtype=float)
    n = len(x)

    if n == 0:
        return 0.0, 0.0
    if n == 1:
        return float(x[0]), 0.0

    mean = float(x.mean())
    standard_error = float(x.std(ddof=1) / np.sqrt(n))
    return mean, 1.96 * standard_error


summary_records = []
for category, specification in CATEGORIES.items():
    values = plot_counts_df.loc[
        plot_counts_df["category"] == category,
        "discovered_cell_types",
    ]
    mean, ci95 = mean_and_ci95(values)

    summary_records.append(
        {
            "category": category,
            "total_cell_types": len(specification["cell_types"]),
            "included_traits": len(values),
            "mean_discovered": mean,
            "ci95_half_width": ci95,
        }
    )

summary_df = pd.DataFrame(summary_records).set_index("category")
summary_df

,total_cell_types,included_traits,mean_discovered,ci95_half_width
category,,,,
Brain,12,13,5.923077,0.606210
Immune,39,12,13.416667,2.669797


## 5. Plot and save the figure

The lighter background bars show the total curated cell types in each category. The darker foreground bars show the mean number associated per included trait; error bars are 95% confidence intervals, and points are individual traits.

In [6]:
categories = list(CATEGORIES)
positions = np.array([1, 2])
bar_width = 0.6
colors = ["lightcoral", "lightblue"]

fig, ax = plt.subplots(figsize=(10, 9))

# Background bars: total curated cell types in each category.
ax.bar(
    positions,
    summary_df.loc[categories, "total_cell_types"],
    width=bar_width,
    color=colors,
    alpha=0.5,
)

# Foreground bars: mean number of associated cell types per included trait.
ax.bar(
    positions,
    summary_df.loc[categories, "mean_discovered"],
    width=bar_width,
    color=colors,
    alpha=0.95,
)

# 95% confidence intervals around the means.
ax.errorbar(
    positions,
    summary_df.loc[categories, "mean_discovered"],
    yerr=summary_df.loc[categories, "ci95_half_width"],
    fmt="none",
    ecolor="black",
    elinewidth=2,
    capsize=10,
)

# Individual trait counts with deterministic horizontal jitter.
rng = np.random.default_rng(JITTER_SEED)
for position, category in zip(positions, categories):
    values = plot_counts_df.loc[
        plot_counts_df["category"] == category,
        "discovered_cell_types",
    ].to_numpy()
    x_values = position + rng.uniform(-0.08, 0.08, size=len(values))
    ax.scatter(x_values, values, color="black", s=45, alpha=0.6, zorder=5)

labels = [
    f"{category} diseases\n(n={summary_df.loc[category, 'included_traits']})"
    for category in categories
]
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=30)
ax.set_ylabel("Number of cell types", fontsize=30)
ax.set_title("Marginal cell type-disease associations", fontsize=34)
ax.tick_params(axis="y", labelsize=16)

legend_handles = [
    Patch(facecolor="0.85", edgecolor="0.85", label="Total cell types"),
    Patch(facecolor="0.45", edgecolor="0.45", label="Associated cell types"),
]
ax.legend(handles=legend_handles, fontsize=20, loc="upper left")

fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved {OUTPUT_FILE.resolve()}")

Saved /mnt/shared-workspace/scdrsfm/nb_run/discovered_cts_boxplot_0.1.png
